# 01. ASM LUT Generation

**v6 Section 17.10.1** | Phase C Development Notebook #1

## Purpose
PINN L_phase target 생성: z=40 (PINN 입사 경계)에서의 복소장 LUT

## Pipeline
```
TMM (AR Gorilla DX 4층) → 초기 평면파 → ASM (CG 550μm 전파) → U(x, z=40, θ)
```

## Output
- `data/asm_luts/incident_z40.npz`

## Reference
- v6 Section 2.4: AR coating specification
- v6 Section 3.2: ASM module role
- v6 Section 3.5.2: TMM → ASM interface
- v6 Section 3.5.3: ASM → PINN interface (LUT format)
- v6 Section 4.3 BC-1: Incident boundary condition

---
## Cell 1: Import & Environment Setup

In [1]:
import sys
from pathlib import Path

# Add project root for backend imports
PROJECT_ROOT = Path.cwd().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from backend.physics.tmm_calculator import GorillaDXTMM, PHASE1_AR_THICKNESSES_NM
from backend.physics.asm_propagator import (
    ASMPropagator, generate_incident_lut,
    CG_THICKNESS_UM, N_CG, WAVELENGTH_UM,
)

print(f"Project root: {PROJECT_ROOT}")
print(f"AR thicknesses (nm): {PHASE1_AR_THICKNESSES_NM}")
print(f"CG: {CG_THICKNESS_UM} μm, n={N_CG}")
print(f"Wavelength: {WAVELENGTH_UM*1000:.0f} nm")

Project root: c:\Users\k2kw2\AI_PINN_PROJECT_v03
AR thicknesses (nm): [34.6, 25.9, 20.7, 169.5]
CG: 550.0 μm, n=1.52
Wavelength: 520 nm


---
## Cell 2: TMM Calculation (AR Gorilla DX 4-layer)

v6 Section 2.4: Phase 1 optimal values
- SiO2(34.6nm) / TiO2(25.9nm) / SiO2(20.7nm) / TiO2(169.5nm)
- Expected: t(0°) ≈ 0.99, Δφ(30°) ≈ -7.5°

In [2]:
# ── TMM LUT 계산 ──
tmm = GorillaDXTMM()  # Phase 1 optimal AR values

# v6 Section 3.5.3: theta range -41 ~ +41 degrees, 1-degree spacing
theta_array = np.arange(-41, 42, 1.0)  # 83 angles
tmm_lut = tmm.compute_lut(theta_array)

print(f"Theta range: [{theta_array[0]}, {theta_array[-1]}] deg, {len(theta_array)} points")
print(f"\nValidation (v6 expected values):")
print(f"  t(0°)  = {tmm_lut['t_amplitude'][41]:.4f}  (expected ~0.99)")
print(f"  Δφ(30°) = {tmm_lut['phase_shift_deg'][71]:.2f}°  (expected ~-7.5°)")
print(f"  t(30°) = {tmm_lut['t_amplitude'][71]:.4f}")

Theta range: [-41.0, 41.0] deg, 83 points

Validation (v6 expected values):
  t(0°)  = 0.8063  (expected ~0.99)
  Δφ(30°) = 5.16°  (expected ~-7.5°)
  t(30°) = 0.7755


In [ ]:
# ── TMM 시각화: 투과율 곡선 & 위상 지연 곡선 ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: Transmission amplitude
ax = axes[0]
ax.plot(tmm_lut['theta_deg'], tmm_lut['t_amplitude'], 'b-', linewidth=1.5)
ax.set_xlabel('Incidence angle (deg)')
ax.set_ylabel('Transmission amplitude |t|')
ax.set_title('AR Coating: Transmission vs Angle')
ax.axhline(y=0.99, color='gray', linestyle='--', alpha=0.5, label='0.99 ref')
ax.axvline(x=0, color='gray', linestyle=':', alpha=0.3)
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim([0.8, 1.05])

# Right: Phase shift
ax = axes[1]
ax.plot(tmm_lut['theta_deg'], tmm_lut['phase_shift_deg'], 'r-', linewidth=1.5)
ax.set_xlabel('Incidence angle (deg)')
ax.set_ylabel('Phase shift Δφ (deg)')
ax.set_title('AR Coating: Phase Shift vs Angle')
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.axvline(x=0, color='gray', linestyle=':', alpha=0.3)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Cell 3: ASM Propagation (CG 550μm)

v6 Section 3.5.3: LUT specification
- x: 0 ~ 504 μm, 4096 points (dx ≈ 0.123 μm)
- theta: -41° ~ +41°, 1° spacing (83 angles)
- Propagation: z=590 → z=40 (550 μm through CG, n=1.52)

In [ ]:
# ── ASM LUT 생성 ──
# v6 Section 3.5.3: N_x = 4096, x in [0, 504] μm
N_X = 4096
x_array = np.linspace(0, 504.0, N_X)  # μm
dx = x_array[1] - x_array[0]

print(f"Spatial grid: {N_X} points, dx = {dx:.4f} μm")
print(f"x range: [{x_array[0]}, {x_array[-1]}] μm")
print(f"Generating LUT for {len(theta_array)} angles...")

lut_data = generate_incident_lut(tmm, theta_array, x_array)

print(f"\nLUT shapes:")
print(f"  theta_values: {lut_data['theta_values'].shape}")
print(f"  x_values:     {lut_data['x_values'].shape}")
print(f"  U_re:         {lut_data['U_re'].shape}")
print(f"  U_im:         {lut_data['U_im'].shape}")
print("Done.")

---
## Cell 4: Complex Field Visualization

v6 Section 17.10.1 검증 시각화:
- |U|(x) at z=40, θ=0° (대칭, 평면파 기대)
- |U|(x) at z=40, θ=30° (비대칭 기대)
- Re/Im heatmap (θ vs x)

In [ ]:
# Helper: get field for a specific angle
def get_field(lut, theta_deg):
    """Extract complex field for a given angle from LUT."""
    idx = np.argmin(np.abs(lut['theta_values'] - theta_deg))
    U = lut['U_re'][idx] + 1j * lut['U_im'][idx]
    return U, lut['theta_values'][idx]

# ── |U|(x) line plots for key angles ──
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for ax, theta_target in zip(axes.flat, [0, 15, 30, 40]):
    U, theta_actual = get_field(lut_data, theta_target)
    amplitude = np.abs(U)
    
    ax.plot(x_array, amplitude, linewidth=0.5)
    ax.set_title(f'|U|(x) at z=40, θ={theta_actual:.0f}°')
    ax.set_xlabel('x (μm)')
    ax.set_ylabel('|U|')
    ax.set_ylim([0, amplitude.max() * 1.2])
    ax.grid(True, alpha=0.3)
    
    # Mark OPD pixel centers (72 μm pitch)
    for i in range(7):
        cx = i * 72 + 36
        ax.axvline(x=cx, color='gray', linestyle=':', alpha=0.2)
    
    # Stats
    ax.text(0.02, 0.95, f'mean={amplitude.mean():.4f}\nstd={amplitude.std():.4f}',
            transform=ax.transAxes, fontsize=8, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

In [ ]:
# ── Re/Im heatmap (θ vs x) ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

extent = [x_array[0], x_array[-1], theta_array[0], theta_array[-1]]

# Real part
ax = axes[0]
im = ax.imshow(lut_data['U_re'], aspect='auto', extent=extent,
               origin='lower', cmap='RdBu_r')
ax.set_title('Re(U) at z=40')
ax.set_xlabel('x (μm)')
ax.set_ylabel('θ (deg)')
plt.colorbar(im, ax=ax, shrink=0.8)

# Imaginary part
ax = axes[1]
im = ax.imshow(lut_data['U_im'], aspect='auto', extent=extent,
               origin='lower', cmap='RdBu_r')
ax.set_title('Im(U) at z=40')
ax.set_xlabel('x (μm)')
ax.set_ylabel('θ (deg)')
plt.colorbar(im, ax=ax, shrink=0.8)

# Amplitude
ax = axes[2]
amplitude_map = np.sqrt(lut_data['U_re']**2 + lut_data['U_im']**2)
im = ax.imshow(amplitude_map, aspect='auto', extent=extent,
               origin='lower', cmap='viridis')
ax.set_title('|U| at z=40')
ax.set_xlabel('x (μm)')
ax.set_ylabel('θ (deg)')
plt.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout()
plt.show()

---
## Cell 5: Verification

v6 Section 17.10.1 검증 항목:
1. θ=0° 수직입사에서 평면파인지 (진폭 균일)
2. 에너지 보존 확인 (ASM 전파 전후)

In [ ]:
print("="*60)
print("VERIFICATION RESULTS")
print("="*60)

# ── Check 1: θ=0 is a plane wave (uniform amplitude) ──
U_0, _ = get_field(lut_data, 0)
amp_0 = np.abs(U_0)
amp_0_std = amp_0.std()
amp_0_mean = amp_0.mean()
is_uniform = amp_0_std / amp_0_mean < 0.01  # < 1% variation

print(f"\n[Check 1] θ=0° plane wave uniformity:")
print(f"  |U| mean = {amp_0_mean:.6f}")
print(f"  |U| std  = {amp_0_std:.6f}")
print(f"  CoV      = {amp_0_std/amp_0_mean*100:.4f}%")
print(f"  Result:  {'PASS' if is_uniform else 'FAIL'} (< 1% variation)")

# ── Check 2: Energy conservation ──
# For a plane wave, |U|^2 integrated over x should be proportional to |t|^2
asm = ASMPropagator()
test_angles = [0, 15, 30, 40]
print(f"\n[Check 2] Energy conservation (ASM propagation):")

all_pass = True
for theta in test_angles:
    tmm_out = tmm.compute(theta)
    U_init = asm.make_initial_field(tmm_out, x_array)
    U_prop = asm.propagate(U_init, dx)
    
    energy_in = np.sum(np.abs(U_init)**2) * dx
    energy_out = np.sum(np.abs(U_prop)**2) * dx
    ratio = energy_out / energy_in
    conserved = abs(ratio - 1.0) < 0.01
    all_pass = all_pass and conserved
    
    print(f"  θ={theta:3d}°: E_in={energy_in:.2f}, E_out={energy_out:.2f}, "
          f"ratio={ratio:.6f}  {'PASS' if conserved else 'FAIL'}")

# ── Check 3: Symmetry at θ=0 ──
phase_0 = np.angle(U_0)
phase_variation = phase_0.std()
is_symmetric = phase_variation < 0.01  # nearly constant phase at θ=0

print(f"\n[Check 3] θ=0° phase symmetry:")
print(f"  Phase std = {phase_variation:.6f} rad")
print(f"  Result:   {'PASS' if is_symmetric else 'FAIL'}")

# ── Summary ──
print(f"\n{'='*60}")
all_ok = is_uniform and all_pass and is_symmetric
print(f"Overall: {'ALL CHECKS PASSED' if all_ok else 'SOME CHECKS FAILED'}")
print(f"{'='*60}")

---
## Cell 6: Save NPZ

v6 Section 3.5.3: `data/asm_luts/incident_z40.npz` format

In [ ]:
# ── Save LUT as NPZ ──
output_path = PROJECT_ROOT / 'data' / 'asm_luts' / 'incident_z40.npz'
output_path.parent.mkdir(parents=True, exist_ok=True)

np.savez(
    output_path,
    theta_values=lut_data['theta_values'],
    x_values=lut_data['x_values'],
    U_re=lut_data['U_re'],
    U_im=lut_data['U_im'],
    # Metadata as separate arrays
    z_inlet=np.float32(40.0),
    cg_thick=np.float32(CG_THICKNESS_UM),
    wavelength_nm=np.float32(520.0),
    n_cg=np.float32(N_CG),
)

file_size_mb = output_path.stat().st_size / 1024 / 1024
print(f"Saved: {output_path}")
print(f"Size:  {file_size_mb:.1f} MB")
print(f"\nContents:")

# Verify by loading back
data = np.load(output_path)
for key in data.files:
    arr = data[key]
    print(f"  {key:20s}: shape={str(arr.shape):15s} dtype={arr.dtype}")
data.close()

print(f"\nLUT generation complete. Ready for PINN L_phase target.")

---
## Summary

| Item | Value |
|------|-------|
| Theta range | -41° ~ +41°, 83 angles |
| X range | 0 ~ 504 μm, 4096 points |
| AR coating | Phase 1 optimal (34.6/25.9/20.7/169.5 nm) |
| CG propagation | 550 μm, n=1.52 |
| Output | `data/asm_luts/incident_z40.npz` |

### Next Step
→ `02_pinn_cpu_validation.ipynb`: 이 LUT를 L_phase target으로 사용하여 PINN 학습 검증